# 企業所有ネットワーク分析
## Corporate Ownership Network Analysis

---

### 研究背景

企業間の株式所有関係は、経済における権力構造やリスク伝播を理解する上で極めて重要です。
Vitali, Glattfelder & Battiston (2011) の画期的な研究では、多国籍企業のグローバルな所有構造を
ネットワーク科学の手法で分析し、経済的権力の著しい集中を明らかにしました。

本ノートブックでは、以下のアプローチで企業所有ネットワークを分析します：

- **Wikidata SPARQL** エンドポイントから企業間の所有関係データを取得
- **GLEIF LEI**（Legal Entity Identifier）データを活用した企業識別
- **NetworkX** を用いた有向グラフとしてのネットワーク構築・分析
- 中心性指標、循環所有（株式持ち合い）の検出

日本の企業グループ（系列）は、相互持合い構造という独特のネットワーク特性を持ち、
ネットワーク科学の観点から特に興味深い研究対象です。

### 関連ドキュメント

詳細な研究サーベイについては、以下のドキュメントを参照してください：

- [コーポレートガバナンス × 知識グラフ × ネットワーク科学](../04-corporate-governance.md)

## 環境セットアップ

必要なライブラリをインストールします。

In [ ]:
# 必要なライブラリのインストール
!pip install networkx pandas matplotlib sparqlwrapper

## ライブラリのインポート

In [ ]:
# 標準ライブラリ
import json
import warnings
warnings.filterwarnings('ignore')

# データ処理
import pandas as pd
import numpy as np

# ネットワーク分析
import networkx as nx

# 可視化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
# 日本語フォントの設定（環境に応じて変更してください）
# macOS の場合
matplotlib.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Yu Gothic', 'Meirio', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

# SPARQL クエリ用
from SPARQLWrapper import SPARQLWrapper, JSON

print("ライブラリのインポートが完了しました。")
print(f"NetworkX バージョン: {nx.__version__}")
print(f"Pandas バージョン: {pd.__version__}")

## Wikidata SPARQL によるデータ取得

Wikidata の SPARQL エンドポイントを使用して、日本の主要企業の所有関係を取得します。

使用するプロパティ：
- **P127** (owned by): 企業の所有者
- **P1268** (represents organization): 組織の代表
- **P17** (country): 国
- **P452** (industry): 業種

Wikidata は世界最大のオープンな知識グラフの一つであり、
企業間の所有関係も含む構造化データを SPARQL で自由に取得できます。

In [ ]:
def fetch_ownership_from_wikidata():
    """
    Wikidata SPARQL エンドポイントから日本企業の所有関係を取得する関数。
    
    Returns:
        pd.DataFrame: 所有関係のDataFrame（company, companyLabel, owner, ownerLabel）
    """
    # Wikidata SPARQL エンドポイント
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    
    # 日本の上場企業の主要株主関係を取得するSPARQLクエリ
    query = """
    SELECT ?company ?companyLabel ?owner ?ownerLabel WHERE {
      # 企業であること（Q4830453: business enterprise）
      ?company wdt:P31/wdt:P279* wd:Q4830453 .
      
      # 日本の企業であること
      ?company wdt:P17 wd:Q17 .
      
      # 所有関係（P127: owned by）
      ?company wdt:P127 ?owner .
      
      # 所有者も組織であること
      ?owner wdt:P31/wdt:P279* wd:Q43229 .
      
      # ラベル取得サービス
      SERVICE wikibase:label { bd:serviceParam wikibase:language "ja,en". }
    }
    LIMIT 500
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        # クエリを実行
        print("Wikidata SPARQL エンドポイントにクエリを送信中...")
        results = sparql.query().convert()
        
        # 結果をリストに変換
        rows = []
        for result in results["results"]["bindings"]:
            rows.append({
                "company": result["company"]["value"],
                "companyLabel": result["companyLabel"]["value"],
                "owner": result["owner"]["value"],
                "ownerLabel": result["ownerLabel"]["value"]
            })
        
        # DataFrameに変換
        df = pd.DataFrame(rows)
        print(f"取得件数: {len(df)} 件の所有関係")
        return df
    
    except Exception as e:
        print(f"Wikidata からのデータ取得に失敗しました: {e}")
        print("サンプルデータを使用します。")
        return None

# データ取得を試行
wikidata_df = fetch_ownership_from_wikidata()

if wikidata_df is not None and len(wikidata_df) > 0:
    print("\n取得データのプレビュー:")
    display(wikidata_df.head(10))

## サンプルデータでの代替

Wikidata SPARQL API が不安定な場合や、取得データが不十分な場合に備えて、
日本の主要企業グループ（三菱グループ、三井グループ、住友グループなど）の
所有関係をサンプルデータとして用意します。

日本の企業グループ（系列）は、戦後の財閥解体後に形成された企業集団であり、
相互株式持合い（循環所有）という独特の所有構造を持っています。

In [ ]:
# 日本の主要企業グループの所有関係サンプルデータ
# 実際の所有比率は簡略化しています

sample_data = {
    "owner": [
        # 三菱グループ内の相互持合い
        "三菱UFJフィナンシャル・グループ", "三菱UFJフィナンシャル・グループ", "三菱UFJフィナンシャル・グループ",
        "三菱商事", "三菱商事", "三菱商事",
        "三菱重工業", "三菱重工業",
        "東京海上ホールディングス", "東京海上ホールディングス",
        "明治安田生命",
        # 三井グループ内の相互持合い
        "三井住友フィナンシャルグループ", "三井住友フィナンシャルグループ", "三井住友フィナンシャルグループ",
        "三井物産", "三井物産", "三井物産",
        "三井不動産",
        "三井住友海上",
        # 住友グループ内の相互持合い
        "住友商事", "住友商事",
        "住友電気工業", "住友電気工業",
        "住友化学",
        "NEC",
        # クロスグループの所有関係
        "三菱UFJフィナンシャル・グループ", "三井住友フィナンシャルグループ",
        "トヨタ自動車", "トヨタ自動車", "トヨタ自動車",
        "日本生命", "日本生命", "日本生命",
        # トヨタグループ
        "豊田自動織機",
        "デンソー",
        "アイシン",
    ],
    "company": [
        # 三菱グループ
        "三菱商事", "三菱重工業", "東京海上ホールディングス",
        "三菱UFJフィナンシャル・グループ", "三菱重工業", "明治安田生命",
        "三菱UFJフィナンシャル・グループ", "三菱商事",
        "三菱UFJフィナンシャル・グループ", "三菱商事",
        "三菱UFJフィナンシャル・グループ",
        # 三井グループ
        "三井物産", "三井不動産", "三井住友海上",
        "三井住友フィナンシャルグループ", "三井不動産", "三井住友海上",
        "三井住友フィナンシャルグループ",
        "三井住友フィナンシャルグループ",
        # 住友グループ
        "住友電気工業", "NEC",
        "住友商事", "住友化学",
        "住友商事",
        "住友電気工業",
        # クロスグループ
        "トヨタ自動車", "トヨタ自動車",
        "三菱UFJフィナンシャル・グループ", "デンソー", "アイシン",
        "三菱UFJフィナンシャル・グループ", "三井住友フィナンシャルグループ", "トヨタ自動車",
        # トヨタグループ
        "トヨタ自動車",
        "トヨタ自動車",
        "トヨタ自動車",
    ],
    "ownership_pct": [
        # 三菱グループ
        3.5, 2.8, 4.1,
        4.2, 1.5, 2.0,
        2.1, 1.8,
        3.0, 2.5,
        3.8,
        # 三井グループ
        3.2, 4.5, 2.8,
        3.8, 2.1, 1.9,
        2.5,
        3.1,
        # 住友グループ
        2.5, 1.8,
        3.0, 2.2,
        2.8,
        1.5,
        # クロスグループ
        2.0, 1.5,
        5.0, 24.0, 7.5,
        3.5, 2.8, 3.2,
        # トヨタグループ
        8.0,
        9.0,
        7.0,
    ],
    "group": [
        # 三菱グループ
        "三菱", "三菱", "三菱",
        "三菱", "三菱", "三菱",
        "三菱", "三菱",
        "三菱", "三菱",
        "三菱",
        # 三井グループ
        "三井", "三井", "三井",
        "三井", "三井", "三井",
        "三井",
        "三井",
        # 住友グループ
        "住友", "住友",
        "住友", "住友",
        "住友",
        "住友",
        # クロスグループ
        "クロス", "クロス",
        "トヨタ", "トヨタ", "トヨタ",
        "クロス", "クロス", "クロス",
        # トヨタグループ
        "トヨタ",
        "トヨタ",
        "トヨタ",
    ]
}

# DataFrameとして構築
df = pd.DataFrame(sample_data)

# Wikidata から取得できた場合はそちらを優先（ただしグループ情報は付与されない）
if wikidata_df is not None and len(wikidata_df) > 10:
    print("Wikidata から取得したデータを使用します。")
    # Wikidata のデータを使用する場合は列名を調整
    df_analysis = wikidata_df.rename(columns={"ownerLabel": "owner", "companyLabel": "company"})
    use_wikidata = True
else:
    print("サンプルデータを使用します。")
    df_analysis = df.copy()
    use_wikidata = False

print(f"\nデータ件数: {len(df_analysis)} 件の所有関係")
print(f"\nデータのプレビュー:")
display(df_analysis.head(10))

## 所有ネットワークの構築

所有関係を NetworkX の有向グラフ（DiGraph）として構築します。

- **ノード**: 各企業
- **有向エッジ**: 所有者 → 被所有企業（株式保有の方向）
- **エッジの重み**: 所有比率（%）

In [ ]:
# 有向グラフの構築
G = nx.DiGraph()

# エッジの追加（所有者 → 被所有企業）
for _, row in df_analysis.iterrows():
    owner = row["owner"]
    company = row["company"]
    
    # エッジ属性を設定
    edge_attrs = {}
    if "ownership_pct" in row:
        edge_attrs["weight"] = row["ownership_pct"]
    if "group" in row:
        edge_attrs["group"] = row["group"]
    
    G.add_edge(owner, company, **edge_attrs)

# サンプルデータ使用時はノード属性（企業グループ）を設定
if not use_wikidata:
    # 各企業が属するグループを特定
    node_groups = {}
    for _, row in df.iterrows():
        group = row["group"]
        node_groups[row["owner"]] = group
        node_groups[row["company"]] = group
    
    # ノード属性として設定
    nx.set_node_attributes(G, node_groups, "group")

print(f"有向グラフを構築しました。")
print(f"ノード数: {G.number_of_nodes()}")
print(f"エッジ数: {G.number_of_edges()}")

## 基本統計量

構築した所有ネットワークの基本的な統計量を計算します。

In [ ]:
# 基本統計量の計算
print("=" * 50)
print("所有ネットワークの基本統計量")
print("=" * 50)

# ノード数とエッジ数
print(f"\nノード数（企業数）: {G.number_of_nodes()}")
print(f"エッジ数（所有関係数）: {G.number_of_edges()}")

# ネットワーク密度
density = nx.density(G)
print(f"ネットワーク密度: {density:.4f}")

# 強連結成分数（すべてのノード間で双方向に到達可能な部分グラフ）
strongly_connected = list(nx.strongly_connected_components(G))
print(f"強連結成分数: {len(strongly_connected)}")

# 弱連結成分数（方向を無視した場合に連結な部分グラフ）
weakly_connected = list(nx.weakly_connected_components(G))
print(f"弱連結成分数: {len(weakly_connected)}")

# 強連結成分の詳細
print(f"\n--- 強連結成分の詳細 ---")
# サイズが2以上の強連結成分（相互持合いを含む）
scc_large = [c for c in strongly_connected if len(c) > 1]
print(f"サイズ2以上の強連結成分数: {len(scc_large)}")
for i, component in enumerate(scc_large):
    print(f"  成分 {i+1} (サイズ {len(component)}): {', '.join(sorted(component))}")

# 平均次数
avg_in_degree = sum(dict(G.in_degree()).values()) / G.number_of_nodes()
avg_out_degree = sum(dict(G.out_degree()).values()) / G.number_of_nodes()
print(f"\n平均入次数: {avg_in_degree:.2f}")
print(f"平均出次数: {avg_out_degree:.2f}")

## ネットワーク可視化

所有ネットワークを可視化します。
企業グループごとに色分けし、エッジの矢印で所有方向を表示します。

In [ ]:
# ネットワークの可視化
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# レイアウトの計算（スプリングレイアウト）
pos = nx.spring_layout(G, k=2.0, iterations=50, seed=42)

# 企業グループごとの色分け
group_colors = {
    "三菱": "#E74C3C",   # 赤
    "三井": "#3498DB",   # 青
    "住友": "#2ECC71",   # 緑
    "トヨタ": "#F39C12", # オレンジ
    "クロス": "#9B59B6", # 紫
}

# ノードの色を決定
node_colors = []
for node in G.nodes():
    group = G.nodes[node].get("group", "その他")
    color = group_colors.get(group, "#95A5A6")  # デフォルトはグレー
    node_colors.append(color)

# ノードサイズを次数に基づいて設定
node_sizes = []
for node in G.nodes():
    degree = G.degree(node)
    node_sizes.append(300 + degree * 200)

# エッジの描画（矢印付き）
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#BDC3C7",
    arrows=True,
    arrowsize=15,
    arrowstyle="->",
    connectionstyle="arc3,rad=0.1",
    alpha=0.6,
    width=1.5
)

# ノードの描画
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.9,
    edgecolors="white",
    linewidths=2
)

# ラベルの描画
nx.draw_networkx_labels(
    G, pos, ax=ax,
    font_size=8,
    font_weight="bold"
)

# 凡例の作成
legend_elements = []
for group_name, color in group_colors.items():
    legend_elements.append(
        plt.scatter([], [], c=color, s=100, label=f"{group_name}グループ")
    )

ax.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=10,
    title="企業グループ",
    title_fontsize=12
)

ax.set_title("日本の主要企業グループ 所有ネットワーク", fontsize=16, fontweight="bold", pad=20)
ax.axis("off")
plt.tight_layout()
plt.show()

## 中心性分析

ネットワーク中心性指標を計算し、各企業の所有ネットワークにおける位置づけを分析します。

- **入次数中心性（In-degree Centrality）**: 多くの企業から所有されている企業（被所有の中心）
- **出次数中心性（Out-degree Centrality）**: 多くの企業を所有している企業（所有の中心）
- **PageRank**: 重要な企業から所有されている企業ほど高いスコア

In [ ]:
# 各種中心性の計算

# 入次数中心性（被所有の中心性）
in_degree_centrality = nx.in_degree_centrality(G)

# 出次数中心性（所有の中心性）
out_degree_centrality = nx.out_degree_centrality(G)

# PageRank
pagerank = nx.pagerank(G, alpha=0.85)

# 結果をDataFrameにまとめる
centrality_df = pd.DataFrame({
    "企業名": list(G.nodes()),
    "入次数中心性": [in_degree_centrality[n] for n in G.nodes()],
    "出次数中心性": [out_degree_centrality[n] for n in G.nodes()],
    "PageRank": [pagerank[n] for n in G.nodes()],
    "入次数": [G.in_degree(n) for n in G.nodes()],
    "出次数": [G.out_degree(n) for n in G.nodes()],
})

# 入次数中心性ランキング（多くの企業から所有されている企業）
print("=" * 60)
print("入次数中心性ランキング（被所有の中心）")
print("多くの企業から株式を保有されている企業")
print("=" * 60)
top_in = centrality_df.sort_values("入次数中心性", ascending=False).head(10)
display(top_in[["企業名", "入次数中心性", "入次数"]].reset_index(drop=True))

# 出次数中心性ランキング（多くの企業を所有している企業）
print("\n" + "=" * 60)
print("出次数中心性ランキング（所有の中心）")
print("多くの企業の株式を保有している企業")
print("=" * 60)
top_out = centrality_df.sort_values("出次数中心性", ascending=False).head(10)
display(top_out[["企業名", "出次数中心性", "出次数"]].reset_index(drop=True))

# PageRank ランキング
print("\n" + "=" * 60)
print("PageRank ランキング")
print("ネットワーク上の重要度（重要な企業から所有されるほど高スコア）")
print("=" * 60)
top_pr = centrality_df.sort_values("PageRank", ascending=False).head(10)
display(top_pr[["企業名", "PageRank"]].reset_index(drop=True))

# 中心性の可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 入次数中心性
top_in_plot = centrality_df.nlargest(8, "入次数中心性")
axes[0].barh(top_in_plot["企業名"], top_in_plot["入次数中心性"], color="#E74C3C")
axes[0].set_title("入次数中心性\n（被所有の中心）", fontsize=12, fontweight="bold")
axes[0].set_xlabel("中心性スコア")
axes[0].invert_yaxis()

# 出次数中心性
top_out_plot = centrality_df.nlargest(8, "出次数中心性")
axes[1].barh(top_out_plot["企業名"], top_out_plot["出次数中心性"], color="#3498DB")
axes[1].set_title("出次数中心性\n（所有の中心）", fontsize=12, fontweight="bold")
axes[1].set_xlabel("中心性スコア")
axes[1].invert_yaxis()

# PageRank
top_pr_plot = centrality_df.nlargest(8, "PageRank")
axes[2].barh(top_pr_plot["企業名"], top_pr_plot["PageRank"], color="#2ECC71")
axes[2].set_title("PageRank", fontsize=12, fontweight="bold")
axes[2].set_xlabel("PageRank スコア")
axes[2].invert_yaxis()

plt.suptitle("中心性分析: 所有ネットワークにおける企業の位置づけ", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 循環所有の検出

有向グラフにおける循環（サイクル）を検出します。
企業間の循環所有は「株式持ち合い」として知られ、日本の企業グループ（系列）の
重要な特徴の一つです。

- 長さ2のサイクル: 二社間の相互持合い（A → B → A）
- 長さ3以上のサイクル: より複雑な循環所有構造

In [ ]:
# 循環所有（株式持ち合い）の検出
print("=" * 60)
print("循環所有（株式持ち合い）の検出")
print("=" * 60)

# すべてのサイクルを検出
cycles = list(nx.simple_cycles(G))

print(f"\n検出されたサイクル数: {len(cycles)}")

# サイクルを長さ別に分類
cycles_by_length = {}
for cycle in cycles:
    length = len(cycle)
    if length not in cycles_by_length:
        cycles_by_length[length] = []
    cycles_by_length[length].append(cycle)

print(f"\nサイクルの長さ別分布:")
for length in sorted(cycles_by_length.keys()):
    count = len(cycles_by_length[length])
    if length == 2:
        label = "二社間の相互持合い"
    elif length == 3:
        label = "三社間の循環所有"
    else:
        label = f"{length}社間の循環所有"
    print(f"  長さ {length} ({label}): {count} 件")

# 各サイクルの詳細表示
print(f"\n--- サイクルの詳細 ---")
for length in sorted(cycles_by_length.keys()):
    print(f"\n【長さ {length} のサイクル】")
    for i, cycle in enumerate(cycles_by_length[length]):
        # サイクルを分かりやすく表示
        cycle_str = " → ".join(cycle) + f" → {cycle[0]}"
        print(f"  {i+1}. {cycle_str}")

# 相互持合いネットワークの可視化（サイクルに関与するエッジのみ）
if len(cycles) > 0:
    print(f"\n\n循環所有に関与する企業の可視化:")
    
    # サイクルに関与するノードとエッジを抽出
    cycle_nodes = set()
    cycle_edges = set()
    for cycle in cycles:
        for i in range(len(cycle)):
            cycle_nodes.add(cycle[i])
            cycle_edges.add((cycle[i], cycle[(i + 1) % len(cycle)]))
    
    # サブグラフを作成
    G_cycles = G.subgraph(cycle_nodes).copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    pos_cycle = nx.spring_layout(G_cycles, k=2.5, iterations=50, seed=42)
    
    # エッジの色分け（サイクルに含まれるエッジは赤、それ以外はグレー）
    edge_colors = []
    edge_widths = []
    for edge in G_cycles.edges():
        if edge in cycle_edges:
            edge_colors.append("#E74C3C")
            edge_widths.append(2.5)
        else:
            edge_colors.append("#BDC3C7")
            edge_widths.append(1.0)
    
    nx.draw_networkx_edges(
        G_cycles, pos_cycle, ax=ax,
        edge_color=edge_colors,
        width=edge_widths,
        arrows=True,
        arrowsize=20,
        arrowstyle="->",
        connectionstyle="arc3,rad=0.15",
        alpha=0.7
    )
    
    nx.draw_networkx_nodes(
        G_cycles, pos_cycle, ax=ax,
        node_color="#F39C12",
        node_size=800,
        alpha=0.9,
        edgecolors="white",
        linewidths=2
    )
    
    nx.draw_networkx_labels(
        G_cycles, pos_cycle, ax=ax,
        font_size=9,
        font_weight="bold"
    )
    
    ax.set_title("循環所有（株式持ち合い）ネットワーク\n赤いエッジがサイクルに含まれる所有関係",
                 fontsize=14, fontweight="bold", pad=20)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("\n循環所有は検出されませんでした。")

## 考察

### 分析結果の解釈

#### 日本の企業グループ（系列）の特徴

本分析から、日本の企業グループ（系列）には以下の特徴的なネットワーク構造が観察されます：

1. **高い相互接続性**: 各企業グループ内で密な所有関係が形成されており、
   強連結成分として検出されます。これは戦後の財閥解体後に形成された系列構造を反映しています。

2. **金融機関の中心的役割**: 三菱UFJフィナンシャル・グループや三井住友フィナンシャルグループなどの
   金融機関が高い入次数・出次数中心性を持ち、グループの「ハブ」として機能しています。
   これは Vitali et al. (2011) がグローバルレベルで発見した金融機関の中心性と一致します。

3. **PageRank の分布**: PageRank スコアは、単純な次数中心性とは異なるランキングを示します。
   重要な企業から所有されている企業がより高いスコアを得るため、
   ネットワーク上の「影響力の連鎖」を反映しています。

#### 株式持ち合いの意味

検出された循環所有（株式持ち合い）は、日本の企業統治の重要な特徴です：

- **安定株主の確保**: 敵対的買収からの防衛メカニズムとして機能
- **長期的関係の維持**: 企業間の取引関係を補完する信頼のシグナル
- **議決権の相互確保**: 経営の安定性を高めるための仕組み
- **近年の解消傾向**: コーポレートガバナンス改革により、政策保有株式の縮減が進行中

#### GLEIF LEI データへの拡張可能性

本分析は以下の方向に拡張可能です：

1. **GLEIF LEI データの統合**: [GLEIF](https://www.gleif.org/) が提供する Legal Entity Identifier (LEI)
   データを活用することで、より正確で包括的な企業識別と所有関係のマッピングが可能になります。
   LEI の Level 2 データ（関係データ）には、直接・最終的な親会社情報が含まれています。

2. **グローバルネットワークへの拡張**: CORPNET 研究グループ（University of Amsterdam）の手法を参考に、
   Orbis データベースなどを活用してグローバルな企業所有ネットワークを構築できます。

3. **時系列分析**: 所有関係の時間的変化を追跡することで、
   持ち合い解消の進展やガバナンス改革の影響を定量的に評価できます。

4. **知識グラフとの統合**: Wikidata などの知識グラフと企業所有データを統合することで、
   業種情報、地理的情報、人的ネットワーク（取締役兼任）などの多次元的な分析が可能になります。

### 参考文献

- Vitali, S., Glattfelder, J. B., & Battiston, S. (2011). The network of global corporate control. *PLoS ONE*, 6(10), e25995.
- Garcia-Bernardo, J., Fichtner, J., Takes, F. W., & Heemskerk, E. M. (2017). Uncovering offshore financial centers. *Scientific Reports*, 7, 6246.
- Fichtner, J., Heemskerk, E. M., & Garcia-Bernardo, J. (2017). Hidden power of the Big Three? *Business and Politics*, 19(2), 298-326.
- Lincoln, J. R., & Gerlach, M. L. (2004). *Japan's Network Economy: Structure, Persistence, and Change*. Cambridge University Press.